In [1]:
#Notebook formatting
from IPython.display import display, HTML
display(HTML("<style>.jp-Cell { margin-left: -50% !important; margin-right: -50% !important; }</style>"))

### Project Process Note - Initial Setup & Data Wrangling Phase

This notebook represents the initial setup and data wrangling phase of the Lending Club credit risk project.

The initial data sets are large and arrived with inconsistent data types, mixed formatting, and many sparse or noisy columns. I loaded the data with Dask, using all columns initially as strings to avoid automatic inference errors and maintain full control over cleaning. 

This required exposure to Dask and learning how to handle inference issues common with large, real-world datasets.

Key work completed in this phase:
- Iteratively diagnosed and resolved repeated dtype mismatch errors across dozens of columns.
- Converted columns (`dti`, `emp_length`, other financial metrics) from string to appropriate numeric types.
- Dropped clearly low-value columns (e.g., `url`, `id`, `desc`, `emp_title`, `zip_code`, `policy_code`, etc.) while preserving columns useful for exploratory analysis. I kept sparse but potentially insightful columns like the hardship and settlement fields.
- Created manageable random samples for testing and validation **Lending Club Workbook.xlsx**
- Adjusted the approach several times to keep the process stable for memory.

The cleaning process took several attempts — I revisited and refined steps as new issues surfaced. This notebook focuses on getting the data into a usable state. 

After wrangling the data into a usable state, I will create additional notebooks for the subsequent project stages, including exploratory analysis and insights, modeling and prediction, and any further extensions as needed.

I used **Grok** as an AI assistant for code suggestions and troubleshooting, similar to collaborating with a senior colleague. However, all strategic decisions regarding what to clean, what to keep, column prioritization, and overall project structure were made by me.

In [2]:
#Setup notes:
#Given that this project uses large data sets, I figured I need to look for extra resources not covered in the course content. So I'll be using Dask for the first time. 

#!pip install jupyter-resource-usage
#!jupyter server extension enable --py jupyter_resource_usage --sys-prefix
#!pip install memory_profiler --quiet
#%load_ext memory_profiler
#%unload_ext memory_profiler

#!pip install ipython-autotime --quiet
#%load_ext autotime
#%unload_ext autotime

In [3]:
%load_ext memory_profiler

In [4]:
%load_ext autotime

time: 55.3 μs (started: 2026-04-17 20:16:34 -05:00)


In [5]:
#Function for memory profiling
def memit_gb():
    from memory_profiler import memory_usage
    mem = memory_usage()[0] / 1024   # convert MiB to GB
    print(f"Usage: {mem:.2f} GB")

memit_gb()

Usage: 0.13 GB
time: 101 ms (started: 2026-04-17 20:16:39 -05:00)


In [6]:

import sys
import platform
import numpy as np
import pandas as pd
import time
import warnings
import dask.dataframe as dd
import gc
sns.set_style("whitegrid")
# Noting libraries to import later:
#import matplotlib.pyplot as plt
#import seaborn as sns


#from sklearn.model_selection import train_test_split
#from sklearn.ensemble import RandomForestClassifier
#from sklearn.linear_model import LogisticRegression
#from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
#from sklearn.preprocessing import StandardScaler, OneHotEncoder
#from sklearn.compose import ColumnTransformer
#from sklearn.pipeline import Pipeline

# Utilities
import warnings
warnings.filterwarnings('ignore')

print('imported')

imported
time: 1.57 s (started: 2026-04-17 20:16:52 -05:00)


In [7]:
#Loaded the raw data with all columns as string (dtype='object') to avoid inference errors, then manually converted key columns to appropriate numeric, further down.
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

dfa = dd.read_csv('LCA_2007-2018.csv', dtype='object')

dfr = dd.read_csv('LCR_2007-2018.csv', 
                  dtype={'member_id': 'float64'})

time: 80.8 ms (started: 2026-04-17 20:16:58 -05:00)


In [8]:
# ================================================
# DATA CLEANING / WRANGLING
# Starting to clean and standardize the data
# (renaming columns, fixing data types, handling missing values, etc.)
# ================================================

time: 415 μs (started: 2026-04-17 20:17:01 -05:00)


In [9]:
dfa.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,<NA>,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,leadman,10+ years,MORTGAGE,55000.0,Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,<NA>,debt_consolidation,Debt consolidation,190xx,PA,5.91,0.0,Aug-2003,675.0,679.0,1.0,30.0,<NA>,7.0,0.0,2765.0,29.7,13.0,w,0.0,0.0,4421.723916800001,4421.72,3600.0,821.72,0.0,0.0,0.0,Jan-2019,122.67,<NA>,Mar-2019,564.0,560.0,0.0,30.0,1.0,Individual,<NA>,<NA>,<NA>,0.0,722.0,144904.0,2.0,2.0,0.0,1.0,21.0,4981.0,36.0,3.0,3.0,722.0,34.0,9300.0,3.0,1.0,4.0,4.0,20701.0,1506.0,37.2,0.0,0.0,148.0,128.0,3.0,3.0,1.0,4.0,69.0,4.0,69.0,2.0,2.0,4.0,2.0,5.0,3.0,4.0,9.0,4.0,7.0,0.0,0.0,0.0,3.0,76.9,0.0,0.0,0.0,178050.0,7746.0,2400.0,13734.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,N,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Cash,N,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,68355089,<NA>,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,Engineer,10+ years,MORTGAGE,65000.0,Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,<NA>,small_business,Business,577xx,SD,16.06,1.0,Dec-1999,715.0,719.0,4.0,6.0,<NA>,22.0,0.0,21470.0,19.2,38.0,w,0.0,0.0,25679.66,25679.66,24700.0,979.66,0.0,0.0,0.0,Jun-2016,926.35,<NA>,Mar-2019,699.0,695.0,0.0,<NA>,1.0,Individual,<NA>,<NA>,<NA>,0.0,0.0,204396.0,1.0,1.0,0.0,1.0,19.0,18005.0,73.0,2.0,3.0,6472.0,29.0,111800.0,0.0,0.0,6.0,4.0,9733.0,57830.0,27.1,0.0,0.0,113.0,192.0,2.0,2.0,4.0,2.0,<NA>,0.0,6.0,0.0,5.0,5.0,13.0,17.0,6.0,20.0,27.0,5.0,22.0,0.0,0.0,0.0,2.0,97.4,7.7,0.0,0.0,314017.0,

time: 974 ms (started: 2026-04-17 20:17:03 -05:00)


In [10]:
dfr.head()

,Amount Requested,Application Date,Loan Title,Risk_Score,Debt-To-Income Ratio,Zip Code,State,Employment Length,Policy Code
0,1000.0,2007-05-26,Wedding Covered but No Honeymoon,693.0,10%,481xx,NM,4 years,0.0
1,1000.0,2007-05-26,Consolidating Debt,703.0,10%,010xx,MA,< 1 year,0.0
2,11000.0,2007-05-27,Want to consolidate my debt,715.0,10%,212xx,MD,1 year,0.0
3,6000.0,2007-05-27,waksman,698.0,38.64%,017xx,MA,< 1 year,0.0
4,1500.0,2007-05-27,mdrigo,509.0,9.43%,209xx,MD,< 1 year,0.0


time: 473 ms (started: 2026-04-17 20:17:04 -05:00)


In [11]:
#Rename Rejected loan columns
%%memit
dfr = dfr.rename(columns={
    "Amount Requested": "amount_requested",
    "Application Date": "application_date",
    "Loan Title": "loan_title",
    "Risk_Score": "risk_score",
    "Debt-To-Income Ratio": "dti",
    "Zip Code": "zip_code",
    "State": "state",
    "Employment Length": "emp_length",
    "Policy Code": "policy_code"
})

peak memory: 330.54 MiB, increment: 0.14 MiB
time: 583 ms (started: 2026-04-17 20:17:08 -05:00)


In [12]:
print("Starting numeric conversion for selected columns...")

numeric_cols = [
    'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 
    'recoveries', 'collection_recovery_fee', 'last_pymnt_amnt',
    'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med',
    'mths_since_last_major_derog', 'policy_code', 'acc_now_delinq', 
    'tot_coll_amt', 'tot_cur_bal', 'open_acc_6m', 'open_act_il', 
    'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il', 'total_bal_il', 
    'il_util', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc', 'all_util', 
    'total_rev_hi_lim', 'inq_fi', 'total_cu_tl', 'inq_last_12m', 
    'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 
    'chargeoff_within_12_mths', 'delinq_amnt', 'mo_sin_old_il_acct', 
    'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 
    'mort_acc', 'mths_since_recent_bc', 'mths_since_recent_bc_dlq', 
    'mths_since_recent_inq', 'mths_since_recent_revol_delinq', 
    'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl', 
    'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 
    'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats', 
    'num_tl_120dpd_2m', 'num_tl_30dpd', 'num_tl_90g_dpd_24m', 
    'num_tl_op_past_12m', 'pct_tl_nvr_dlq', 'percent_bc_gt_75', 
    'pub_rec_bankruptcies', 'tax_liens', 'tot_hi_cred_lim', 
    'total_bal_ex_mort', 'total_bc_limit', 'total_il_high_credit_limit',
    'revol_bal_joint', 'sec_app_fico_range_low', 'sec_app_fico_range_high',
    'sec_app_inq_last_6mths', 'sec_app_mort_acc', 'sec_app_open_acc',
    'sec_app_revol_util', 'sec_app_open_act_il', 'sec_app_num_rev_accts',
    'sec_app_chargeoff_within_12_mths', 'sec_app_collections_12_mths_ex_med',
    'sec_app_mths_since_last_major_derog', 'deferral_term', 'hardship_amount',
    'hardship_length', 'hardship_dpd', 'orig_projected_additional_accrued_interest',
    'hardship_payoff_balance_amount', 'hardship_last_payment_amount',
    'settlement_amount', 'settlement_percentage', 'settlement_term'
]

for col in numeric_cols:
    if col in dfa.columns:
        dfa[col] = dfa[col].map_partitions(
            lambda x: pd.to_numeric(x, errors='coerce'), 
            meta=(col, 'float64')
        )

print("Converted")

Starting numeric conversion for selected columns...
Converting numbers
Remaining columns: 151
time: 779 ms (started: 2026-04-17 20:17:11 -05:00)


In [ ]:
print("Converting DFA columns to different types")

# We likely need to convert most of our columns from string[pyarrow] types over to the type of value they're supposed to represent. I left float64 blank for now because I'm not sure yet which numbers make the most sense to convert to float64. 
float64s = []#convert to float64

float32s = ['dti',
'il_util',
'bc_util',
'pct_tl_nvr_dlq',
'percent_bc_gt_75',
'settlement_percentage',
'revol_util']#convert to float32

#Need to convert these to floats, but add padded zeros like '0.00' with :.2f
currencies_decimals = ['orig_projected_additional_accrued_interest',
'revol_bal',
'loan_amnt',
'int_rate',
'installment',
'annual_inc',
'out_prncp',
'out_prncp_inv',
'total_pymnt_inv',
'total_rec_prncp',
'total_rec_int',
'total_rec_late_fee',
'recoveries',
'collection_recovery_fee',
'last_pymnt_amnt',
'annual_inc_joint',
'dti_joint',
'tot_coll_amt',
'tot_cur_bal',
'total_bal_il',
'max_bal_bc',
'total_rev_hi_lim',
'avg_cur_bal',
'total_bal_ex_mort',
'total_bc_limit',
'total_il_high_credit_limit',
'revol_bal_joint',
'hardship_amount',
'hardship_payoff_balance_amount',
'hardship_last_payment_amount',
'settlement_amount',
'total_pymnt'] # I believe these all need two padded zeros since they're currencies, or because they're percentages, or it just makes sense for another functional reason. 

ints = ['delinq_2yrs',
'fico_range_low',
'fico_range_high',
'inq_last_6mths',
'mths_since_last_delinq',
'mths_since_last_record',
'open_acc',
'pub_rec',
'total_acc',
'collections_12_mths_ex_med',
'mths_since_last_major_derog',
'policy_code',
'acc_now_delinq',
'open_acc_6m',
'open_act_il',
'open_il_12m',
'open_il_24m',
'mths_since_rcnt_il',
'open_rv_12m',
'open_rv_24m',
'inq_fi',
'total_cu_tl',
'inq_last_12m',
'acc_open_past_24mths',
'chargeoff_within_12_mths',
'delinq_amnt',
'mo_sin_old_il_acct',
'mo_sin_old_rev_tl_op',
'mo_sin_rcnt_rev_tl_op',
'mo_sin_rcnt_tl',
'mort_acc',
'mths_since_recent_bc',
'mths_since_recent_bc_dlq',
'mths_since_recent_inq',
'mths_since_recent_revol_delinq',
'num_accts_ever_120_pd',
'num_actv_bc_tl',
'num_actv_rev_tl',
'num_bc_sats',
'num_bc_tl',
'num_il_tl',
'num_op_rev_tl',
'num_rev_accts',
'num_rev_tl_bal_gt_0',
'num_sats',
'num_tl_120dpd_2m',
'num_tl_30dpd',
'num_tl_90g_dpd_24m',
'num_tl_op_past_12m',
'pub_rec_bankruptcies',
'tax_liens',
'sec_app_inq_last_6mths',
'sec_app_mort_acc',
'sec_app_open_acc',
'sec_app_revol_util',
'sec_app_open_act_il',
'sec_app_num_rev_accts',
'sec_app_chargeoff_within_12_mths',
'sec_app_collections_12_mths_ex_med',
'sec_app_mths_since_last_major_derog',
'deferral_term',
'hardship_length',
'hardship_dpd',
'settlement_term']#Convert to integers

#For dates conversions on the columns below, all date records besides emp_length appear to be strings like 'Apr-2010', etc. So we need to do some kind of string conversion to turn them into date values. I also need to determine which columns below need to be represented as beginning of the month versus end of a month. 
#Both the dates_bom and dates_eom columns need to have distinct column names in either variable. Currently they have duplicated column names. So delete appropriate column names from either variable. 
dates_bom =[
'issue_d',
'earliest_cr_line',
'last_pymnt_d',
'next_pymnt_d',
'last_credit_pull_d',
'sec_app_earliest_cr_line',
'hardship_start_date',
'hardship_end_date',
'payment_plan_start_date',
'debt_settlement_flag_date',
'settlement_date'] #Convert to beginning of month dates

dates_eom =[
'issue_d',
'earliest_cr_line',
'last_pymnt_d',
'next_pymnt_d',
'last_credit_pull_d',
'sec_app_earliest_cr_line',
'hardship_start_date',
'hardship_end_date',
'payment_plan_start_date',
'debt_settlement_flag_date',
'settlement_date'] #Convert to end of month dates. 


for col in float32s:
    if col in dfa.columns:
        dfa[col] = dfa[col].map_partitions(lambda x: pd.to_numeric(x, errors='coerce'), meta=(col, 'float32'))

print("Converted")
#In lower cells, I already have code blocks for converting the 'term' column, emp_length, and dti. Since those columns need specific treatment like regex or string manipulation. 

""" These columns I'm not sure how to treat just yet:
'last_fico_range_high',
'last_fico_range_low',
'all_util',
'bc_open_to_buy',
'sec_app_fico_range_low',
'sec_app_fico_range_high',
'tot_hi_cred_lim',

"""

In [13]:
#Deleting some columns that are mostly redundant, useless, have sub-categories or more granular useful data in other columns. etc...
dfa = dfa.drop(columns=[
    'url', 
    'member_id', 
    'id', 
    'desc', 
    'title', 
    'emp_title', 
    'zip_code', 
    'policy_code',
    'pymnt_plan',
    'initial_list_status',
    'disbursement_method',
    'funded_amnt',
    'funded_amnt_inv',
    'grade',
    
])

time: 10.8 ms (started: 2026-04-17 20:17:14 -05:00)


In [14]:
#Convert DTI string to numeric float
#Maybe we can get rid of the dfa[dti] line here and include it in the above. The dfr[dti] should stay though. 
dfa['dti'] = dfa['dti'].map_partitions(lambda x: pd.to_numeric(x, errors='coerce'), meta=('dti', 'float64'))
dfr['dti'] = dfr['dti'].astype(str).str.replace('%', '').str.strip()
dfr['dti'] = dfr['dti'].map_partitions(lambda x: pd.to_numeric(x, errors='coerce') / 100, meta=('dti', 'float64'))

print("dti complete")
print("\n dfa [dti] head:")
print(dfa['dti'].head(10))
print("\n dfr [dti] head:")
print(dfr['dti'].head(10))

dti complete

 dfa [dti] head:
0     5.91
1    16.06
2    10.78
3    17.06
4    25.37
5     10.2
6    14.67
7    17.61
8    13.07
9     34.8
Name: dti, dtype: Float64

 dfr [dti] head:
0    0.1000
1    0.1000
2    0.1000
3    0.3864
4    0.0943
5    0.0000
6    0.1000
7    0.1000
8    0.1000
9    0.1176
Name: dti, dtype: float64
time: 19.3 s (started: 2026-04-17 20:17:19 -05:00)


In [15]:
#Check data types
display(dfr.dtypes, dfa.dtypes)

amount_requested            float64
application_date    string[pyarrow]
loan_title          string[pyarrow]
risk_score                  float64
dti                         float64
zip_code            string[pyarrow]
state               string[pyarrow]
emp_length          string[pyarrow]
policy_code                 float64
dtype: object

loan_amnt                                     string[pyarrow]
term                                          string[pyarrow]
int_rate                                      string[pyarrow]
installment                                   string[pyarrow]
sub_grade                                     string[pyarrow]
emp_length                                    string[pyarrow]
home_ownership                                string[pyarrow]
annual_inc                                    string[pyarrow]
verification_status                           string[pyarrow]
issue_d                                       string[pyarrow]
loan_status                                   string[pyarrow]
purpose                                       string[pyarrow]
addr_state                                    string[pyarrow]
dti                                                   float64
delinq_2yrs                                   string[pyarrow]
earliest_cr_line                              string[pyarrow]
fico_ran

time: 3.48 ms (started: 2026-04-17 20:17:40 -05:00)


In [16]:
#Cleaning/converting a few more columns.

dfa['emp_length'] = dfa['emp_length'].astype(str)
dfa['emp_length'] = dfa['emp_length'].str.extract(r'(\d+)')[0].astype(int)
dfa['emp_length'] = dfa['emp_length'].fillna(0)

dfr['emp_length'] = dfr['emp_length'].astype(str)
dfr['emp_length'] = dfr['emp_length'].str.extract(r'(\d+)')[0].astype(int)
dfr['emp_length'] = dfr['emp_length'].fillna(0)

time: 42.7 ms (started: 2026-04-17 20:17:52 -05:00)


In [17]:
dfa['emp_length'].head(10)

0    10
1    10
2    10
3    10
4     3
5     4
6    10
7    10
8     6
9    10
Name: emp_length, dtype: int64

time: 19.2 s (started: 2026-04-17 20:17:54 -05:00)


In [18]:
#Memory check
import gc
gc.collect()

print("Current memory usage after cleanup:")
memit_gb()

print("\nCurrent number of columns in dfa:", len(dfa.columns))
print("Current number of columns in dfr:", len(dfr.columns))

Current memory usage after cleanup:
Usage: 0.46 GB

Current number of columns in dfa: 137
Current number of columns in dfr: 9
time: 170 ms (started: 2026-04-17 20:29:26 -05:00)


In [19]:
dfa.head()

,loan_amnt,term,int_rate,installment,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,purpose,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,mths_since_last_major_derog,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,3600.0,36 months,13.99,123.03,C4,10,MORTGAGE,55000.0,Not Verified,Dec-2015,Fully Paid,debt_consolidation,PA,5.91,0.0,Aug-2003,675.0,679.0,1.0,30.0,<NA>,7.0,0.0,2765.0,29.7,13.0,0.0,0.0,4421.723916800001,4421.72,3600.0,821.72,0.0,0.0,0.0,Jan-2019,122.67,<NA>,Mar-2019,564.0,560.0,0.0,30.0,Individual,<NA>,<NA>,<NA>,0.0,722.0,144904.0,2.0,2.0,0.0,1.0,21.0,4981.0,36.0,3.0,3.0,722.0,34.0,9300.0,3.0,1.0,4.0,4.0,20701.0,1506.0,37.2,0.0,0.0,148.0,128.0,3.0,3.0,1.0,4.0,69.0,4.0,69.0,2.0,2.0,4.0,2.0,5.0,3.0,4.0,9.0,4.0,7.0,0.0,0.0,0.0,3.0,76.9,0.0,0.0,0.0,178050.0,7746.0,2400.0,13734.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,N,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,N,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,24700.0,36 months,11.99,820.28,C1,10,MORTGAGE,65000.0,Not Verified,Dec-2015,Fully Paid,small_business,SD,16.06,1.0,Dec-1999,715.0,719.0,4.0,6.0,<NA>,22.0,0.0,21470.0,19.2,38.0,0.0,0.0,25679.66,25679.66,24700.0,979.66,0.0,0.0,0.0,Jun-2016,926.35,<NA>,Mar-2019,699.0,695.0,0.0,<NA>,Individual,<NA>,<NA>,<NA>,0.0,0.0,204396.0,1.0,1.0,0.0,1.0,19.0,18005.0,73.0,2.0,3.0,6472.0,29.0,111800.0,0.0,0.0,6.0,4.0,9733.0,57830.0,27.1,0.0,0.0,113.0,192.0,2.0,2.0,4.0,2.0,<NA>,0.0,6.0,0.0,5.0,5.0,13.0,17.0,6.0,20.0,27.0,5.0,22.0,0.0,0.0,0.0,2.0,97.4,7.7,0.0,0.0,314017.0,39475.0,79300.0,24667.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,N,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,N,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,20000.0,60 months,10.78,432.66,B4,10,MORTGAGE,63000.0,Not Verified,Dec-2015,Fully Paid,home_improvement,IL,10.78,0.0,Aug-2000,695.0,699.0,0.0,<NA>,<NA>,6.0,0.0,7869.0,56.2,18.0,0.0,0.0,22705.924293878397,22705.92,

time: 21 s (started: 2026-04-17 20:29:28 -05:00)


In [20]:
#Memory check:
import gc
import gc
gc.collect()
print("MEMORY: ")
memit_gb() 

MEMORY usage: :
Usage: 0.55 GB
time: 165 ms (started: 2026-04-17 20:32:20 -05:00)


In [21]:
print("Cleaning term column:")

if 'term' in dfa.columns:
    dfa['term'] = dfa['term'].astype(str).str.extract(r'(\d+)')[0].astype('Int32')
    print("term cleaned on dfa")
else:
    print("term column not found in dfa")

print("Complete")

Cleaning term column:
term cleaned on dfa
Complete
time: 8.7 ms (started: 2026-04-17 20:32:30 -05:00)


In [22]:
print('Converting int_rate to float64') 
dfa['int_rate'] = dfa['int_rate'].map_partitions(lambda x: pd.to_numeric(x, errors='coerce'), meta=('int_rate', 'float64'))
print('complete')

Converting int_rate to float64
complete
time: 9.14 ms (started: 2026-04-17 20:32:34 -05:00)


In [23]:
print('Converting annual_inc to float64') 
dfa['annual_inc'] = dfa['annual_inc'].map_partitions(lambda x: pd.to_numeric(x, errors='coerce'), meta=('annual_inc', 'float64'))
print('complete')

Converting annual_inc to float64
complete
time: 9.13 ms (started: 2026-04-17 20:32:42 -05:00)


In [24]:
print('Converting revol_util to float64') 
dfa['revol_util'] = dfa['revol_util'].map_partitions(lambda x: pd.to_numeric(x, errors='coerce'), meta=('revol_util', 'float64'))
print('complete')

Converting revol_util to float64
complete
time: 8.94 ms (started: 2026-04-17 20:32:44 -05:00)


In [26]:
#Memory check:
import gc
gc.collect()
print("MEMORY: ")
memit_gb() 

MEMORY: 
Usage: 0.54 GB
time: 175 ms (started: 2026-04-17 20:33:18 -05:00)


In [33]:
#Converting more columns to numerics
numeric_cols2 = ['dti','emp_length','term','int_rate','annual_inc','revol_util']
for col in numeric_cols2:
    if col in dfa.columns:
        dfa[col] = dfa[col].fillna(0) 
print('complete')

complete
time: 40 ms (started: 2026-04-17 21:12:28 -05:00)


In [34]:
#Fill NaNs / <NA>
for col in numeric_cols:
    if col in dfa.columns:
        dfa[col] = dfa[col].fillna(0) 
print('complete')

complete
time: 442 ms (started: 2026-04-17 21:12:53 -05:00)


In [35]:
dfa.head(10)

,loan_amnt,term,int_rate,installment,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,purpose,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,mths_since_last_major_derog,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,3600.0,36,13.99,123.03,C4,10,MORTGAGE,55000.0,Not Verified,Dec-2015,Fully Paid,debt_consolidation,PA,5.91,0.0,Aug-2003,675.0,679.0,1.0,30.0,<NA>,7.0,0.0,2765.0,29.7,13.0,0.0,0.0,4421.723916800001,4421.72,3600.0,821.72,0.0,0.0,0.0,Jan-2019,122.67,<NA>,Mar-2019,564.0,560.0,0.0,30.0,Individual,<NA>,<NA>,<NA>,0.0,722.0,144904.0,2.0,2.0,0.0,1.0,21.0,4981.0,36.0,3.0,3.0,722.0,34.0,9300.0,3.0,1.0,4.0,4.0,20701.0,1506.0,37.2,0.0,0.0,148.0,128.0,3.0,3.0,1.0,4.0,69.0,4.0,69.0,2.0,2.0,4.0,2.0,5.0,3.0,4.0,9.0,4.0,7.0,0.0,0.0,0.0,3.0,76.9,0.0,0.0,0.0,178050.0,7746.0,2400.0,13734.0,0.0,0.0,0.0,<NA>,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,N,<NA>,<NA>,<NA>,0.0,0.0,<NA>,<NA>,<NA>,0.0,0.0,<NA>,0.0,0.0,0.0,N,<NA>,<NA>,<NA>,0.0,0.0,0.0
1,24700.0,36,11.99,820.28,C1,10,MORTGAGE,65000.0,Not Verified,Dec-2015,Fully Paid,small_business,SD,16.06,1.0,Dec-1999,715.0,719.0,4.0,6.0,<NA>,22.0,0.0,21470.0,19.2,38.0,0.0,0.0,25679.66,25679.66,24700.0,979.66,0.0,0.0,0.0,Jun-2016,926.35,<NA>,Mar-2019,699.0,695.0,0.0,0.0,Individual,<NA>,<NA>,<NA>,0.0,0.0,204396.0,1.0,1.0,0.0,1.0,19.0,18005.0,73.0,2.0,3.0,6472.0,29.0,111800.0,0.0,0.0,6.0,4.0,9733.0,57830.0,27.1,0.0,0.0,113.0,192.0,2.0,2.0,4.0,2.0,0.0,0.0,6.0,0.0,5.0,5.0,13.0,17.0,6.0,20.0,27.0,5.0,22.0,0.0,0.0,0.0,2.0,97.4,7.7,0.0,0.0,314017.0,39475.0,79300.0,24667.0,0.0,0.0,0.0,<NA>,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,N,<NA>,<NA>,<NA>,0.0,0.0,<NA>,<NA>,<NA>,0.0,0.0,<NA>,0.0,0.0,0.0,N,<NA>,<NA>,<NA>,0.0,0.0,0.0
2,20000.0,60,10.78,432.66,B4,10,MORTGAGE,63000.0,Not Verified,Dec-2015,Fully Paid,home_improvement,IL,10.78,0.0,Aug-2000,695.0,699.0,0.0,<NA>,<NA>,6.0,0.0,7869.0,56.2,18.0,0.0,0.0,22705.924293878397,22705.92,20000.0,2705.92,0.0,0.0,0.0,Jun-2017,15813.3,<NA>,Mar-2019,704.0,70

time: 38.4 s (started: 2026-04-17 21:16:23 -05:00)


In [36]:
numeric_cols3 = ['annual_inc_joint']
for col in numeric_cols3:
    if col in dfa.columns:
        dfa[col] = dfa[col].fillna(0) 
print('complete')

complete
time: 8.22 ms (started: 2026-04-17 21:45:32 -05:00)


In [37]:
dfa['annual_inc_joint'].head()

0          0
1          0
2    71000.0
3          0
4          0
Name: annual_inc_joint, dtype: object

time: 38.1 s (started: 2026-04-17 21:46:13 -05:00)


In [38]:
dfa.head()

,loan_amnt,term,int_rate,installment,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,purpose,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,mths_since_last_major_derog,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,3600.0,36,13.99,123.03,C4,10,MORTGAGE,55000.0,Not Verified,Dec-2015,Fully Paid,debt_consolidation,PA,5.91,0.0,Aug-2003,675.0,679.0,1.0,30.0,<NA>,7.0,0.0,2765.0,29.7,13.0,0.0,0.0,4421.723916800001,4421.72,3600.0,821.72,0.0,0.0,0.0,Jan-2019,122.67,<NA>,Mar-2019,564.0,560.0,0.0,30.0,Individual,0,<NA>,<NA>,0.0,722.0,144904.0,2.0,2.0,0.0,1.0,21.0,4981.0,36.0,3.0,3.0,722.0,34.0,9300.0,3.0,1.0,4.0,4.0,20701.0,1506.0,37.2,0.0,0.0,148.0,128.0,3.0,3.0,1.0,4.0,69.0,4.0,69.0,2.0,2.0,4.0,2.0,5.0,3.0,4.0,9.0,4.0,7.0,0.0,0.0,0.0,3.0,76.9,0.0,0.0,0.0,178050.0,7746.0,2400.0,13734.0,0.0,0.0,0.0,<NA>,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,N,<NA>,<NA>,<NA>,0.0,0.0,<NA>,<NA>,<NA>,0.0,0.0,<NA>,0.0,0.0,0.0,N,<NA>,<NA>,<NA>,0.0,0.0,0.0
1,24700.0,36,11.99,820.28,C1,10,MORTGAGE,65000.0,Not Verified,Dec-2015,Fully Paid,small_business,SD,16.06,1.0,Dec-1999,715.0,719.0,4.0,6.0,<NA>,22.0,0.0,21470.0,19.2,38.0,0.0,0.0,25679.66,25679.66,24700.0,979.66,0.0,0.0,0.0,Jun-2016,926.35,<NA>,Mar-2019,699.0,695.0,0.0,0.0,Individual,0,<NA>,<NA>,0.0,0.0,204396.0,1.0,1.0,0.0,1.0,19.0,18005.0,73.0,2.0,3.0,6472.0,29.0,111800.0,0.0,0.0,6.0,4.0,9733.0,57830.0,27.1,0.0,0.0,113.0,192.0,2.0,2.0,4.0,2.0,0.0,0.0,6.0,0.0,5.0,5.0,13.0,17.0,6.0,20.0,27.0,5.0,22.0,0.0,0.0,0.0,2.0,97.4,7.7,0.0,0.0,314017.0,39475.0,79300.0,24667.0,0.0,0.0,0.0,<NA>,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,N,<NA>,<NA>,<NA>,0.0,0.0,<NA>,<NA>,<NA>,0.0,0.0,<NA>,0.0,0.0,0.0,N,<NA>,<NA>,<NA>,0.0,0.0,0.0
2,20000.0,60,10.78,432.66,B4,10,MORTGAGE,63000.0,Not Verified,Dec-2015,Fully Paid,home_improvement,IL,10.78,0.0,Aug-2000,695.0,699.0,0.0,<NA>,<NA>,6.0,0.0,7869.0,56.2,18.0,0.0,0.0,22705.924293878397,22705.92,20000.0,2705.92,0.0,0.0,0.0,Jun-2017,15813.3,<NA>,Mar-2019,704.0,700.0,0.

time: 38.5 s (started: 2026-04-17 21:47:04 -05:00)


In [ ]:
#TO DO: Quick test on annual_inc_joint worked even though it was a string. 
#There are more columns that need fillna. 
#Revist later and convert a portion of strings to ints, floats, date values.
#Determine a reasonably correct default date value for certain columns (ie... BOM vs EOM)
#Fill the remaining NaN/<NA> with 0 or 0.0. 
# Determine a way to convert currency values from x.0 to x.00

In [ ]:
# ===================================================================
# UTILITIES / SCRATCHPAD
# Random one-liners and helper code I might want later.
# Keep the main notebook clean.
# ===================================================================

# Quick memory cleanup
import gc
gc.collect()

# Delete large objects when done with them
# del dfa, dfr, dfa_sample, dfr_sample

# Example exports (uncomment when needed)
# dfa_sample.to_excel('LCA_10k.xlsx', index=False)
# dfr_sample.to_excel('LCR_10k.xlsx', index=False)

# Quick inspection examples
# dfa.head(20)
# dfa['loan_status'].value_counts().compute()
# dfr.head(10)
# ===================================================================
# MEMORY TEST SNIPPETS - dfa and dfr - column cleaning tests
# [using dti column for test example]
# ===================================================================

# Test 0.1% ----------------------------------------------------------------
testdfa1 = dfa.sample(frac=0.001, random_state=42).compute()
testdfa1['dti'] = testdfa1['dti'].str.replace('%', '').astype(float).round(2)
print("0.1\'%' - dfa - ")
del testdfa1
gc.collect()
# Test 2% ----------------------------------------------------------------
testdfa2 = dfa.sample(frac=0.02, random_state=42).compute()
testdfa2['dti'] = testdfa2['dti'].str.replace('%', '').astype(float).round(2)
print("2\'%' - dfa - ")
del testdfa2
gc.collect()
# Test 10% ----------------------------------------------------------------
testdfa3 = dfa.sample(frac=0.1, random_state=42).compute()
testdfa3['dti'] = testdfa3['dti'].str.replace('%', '').astype(float).round(2)
print("10\'%' - dfa - ")
del testdfa3
gc.collect()
# Test 20% ----------------------------------------------------------------
testdfa4 = dfa.sample(frac=0.2, random_state=42).compute()
testdfa4['dti'] = testdfa4['dti'].str.replace('%', '').astype(float).round(2)
print("20\'%' - dfa - ")
del testdfa4
gc.collect()
# Test 50% ----------------------------------------------------------------
testdfa5 = dfa.sample(frac=0.5, random_state=42).compute()
testdfa5['dti'] = testdfa5['dti'].str.replace('%', '').astype(float).round(2)
print("50\'%' - dfa - ")
del testdfa5
gc.collect()

# Junk for later:
head_df = dfa.head(10)
display(head_df)
print(f"Time taken: {clock} seconds")



